# Flight Booking Decision Assistant — separate practice copy

The working `CS664_Flight_Booking.ipynb` remains unchanged, including its actual outputs. This copy preserves all imports, setup, reference code, and output formatting. Outputs in this copy are cleared so they cannot be mistaken for your execution. Exports from this copy go to `practice_results/`, preserving the official results.

**Active topic: 1 only.** Select **Python (CS664 Flight Booking)**. Run the reference setup cells **1A, 2A, and 2B**, then work on **P1-TODO**. Run **P1-CHECK** only after completing your attempt. Stop there; discuss your attempt with the assistant before moving on.

The other topics will be introduced one at a time: Q updates and terminal handling; epsilon-greedy actions; fair evaluation and a threshold experiment. Submission analyses are tracked separately in `SUBMISSION_FEATURES.md`.

**Recorded official results:** mean costs $239.58 (immediate), $223.476 ($120 rule), $219.738 ($180 rule), and $214.854 (Q-learning). All simulator checks passed. These are the agent's execution results, not evidence of your personal practice.

# Flight Booking Decision Assistant — executed core

This Notebook implements the existing proposal using hypothetical fares, not real airfare data. Maximizing the negative purchase reward minimizes cost. Run cells in order with the **Python (CS664 Flight Booking)** kernel.

The core includes simulation, tabular Q-learning, a fair four-strategy comparison, and a complete decision trace. Confidence intervals, exact MDP comparison, and the policy heatmap are deferred. The original proposal remains unchanged.

## 1. Setup and fixed experiment settings
**Purpose:** make the official experiment reproducible. States are `(time remaining, price)`; table columns represent BOOK and WAIT. Training and test randomness are independent.

In [ ]:
# 1A — imports and official settings
from pathlib import Path
import csv
import json
import sys
import numpy as np
from IPython.display import display, Markdown

PRICES = np.array([120, 180, 240, 300, 360], dtype=int)
BOOK, WAIT = 0, 1
ACTION_NAMES = {BOOK: "BOOK", WAIT: "WAIT"}
TRAIN_EPISODES, TEST_TRAJECTORIES = 50_000, 10_000
ALPHA, GAMMA = 0.1, 1.0
TRAIN_SEED, TEST_SEED = 66401, 66402
OFFICIAL_THRESHOLD = 180
assert TRAIN_SEED != TEST_SEED
print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Training/test seeds:", TRAIN_SEED, TEST_SEED)

## 2. Price simulator
**Purpose:** encode exactly seven opportunities, time 6 through 0. The probabilities depend on the time **before** WAIT. An attempted move past a boundary stays at that boundary. At time 0, any requested action becomes BOOK. After booking, further steps raise an error.

Only the environment knows the transition probabilities. The learner below receives states and observed rewards.

In [ ]:
# 2A — one price transition
def next_price(time_remaining, price, rng):
    if time_remaining not in range(1, 7) or price not in PRICES:
        raise ValueError("WAIT requires time 1–6 and a valid price")
    probabilities = (0.3, 0.3, 0.4) if time_remaining >= 3 else (0.6, 0.2, 0.2)
    change = int(rng.choice([60, 0, -60], p=probabilities))
    return int(np.clip(price + change, PRICES[0], PRICES[-1]))

In [ ]:
# 2B — episode and terminal rules
class FlightBookingEnv:
    def __init__(self, rng):
        self.rng = rng
        self.state = None
        self.done = True

    def reset(self):
        self.state = (6, int(self.rng.choice(PRICES)))
        self.done = False
        return self.state

    def step(self, action):
        if self.done:
            raise RuntimeError("Reset before acting; a purchase ends the episode")
        if action not in (BOOK, WAIT):
            raise ValueError("Action must be BOOK or WAIT")
        time_remaining, price = self.state
        effective_action = BOOK if time_remaining == 0 else action
        if effective_action == BOOK:
            self.done = True
            return None, -price, True, effective_action
        self.state = (time_remaining - 1, next_price(time_remaining, price, self.rng))
        return self.state, 0, False, effective_action

## Practice topic 1 — transitions, rewards, and termination

**Purpose:** understand how each action changes state and reward, and why every episode ends after a purchase. WAIT advances one day with reward 0 and a bounded next price. BOOK pays the current fare (negative reward) and ends the episode. At time 0 the environment must enforce booking.

**Predict before running:** starting at `(0, 180)`, what four values will `env.step(WAIT)` return: next state, reward, done, and effective action?

**Small TODO:** fill only the `effective_action` assignment in **P1-TODO**. It must preserve a requested action before the deadline and enforce the deadline rule. Everything surrounding that line is provided. This practice class is separate from the reference environment above.

Send your predicted result and your one-line attempt to the assistant. A hint comes before a worked solution. Your practice status is **not yet attempted**.

In [ ]:
# P1-TODO — complete one line; leave the surrounding code intact.
class PracticeBookingEnv(FlightBookingEnv):
    def step(self, action):
        if self.done:
            raise RuntimeError("Reset before acting; a purchase ends the episode")
        if action not in (BOOK, WAIT):
            raise ValueError("Action must be BOOK or WAIT")
        time_remaining, price = self.state

        # TODO P1: enforce deadline booking; otherwise use the requested action.
        effective_action = ...

        if effective_action is Ellipsis:
            raise NotImplementedError("Complete TODO P1 before running the check")
        if effective_action == BOOK:
            self.done = True
            return None, -price, True, effective_action
        self.state = (time_remaining - 1, next_price(time_remaining, price, self.rng))
        return self.state, 0, False, effective_action

In [ ]:
# P1-CHECK — run after completing and rerunning P1-TODO.
class DecreaseOnlyRNG:
    def choice(self, values, p=None):
        return 120 if p is None else -60

def check_topic_1():
    env = PracticeBookingEnv(DecreaseOnlyRNG())
    env.reset()
    env.state = (1, 120)
    state, reward, done, action = env.step(WAIT)
    assert state == (0, 120), "WAIT must advance time and respect the lower bound"
    assert reward == 0 and not done and action == WAIT
    terminal, reward, done, action = env.step(WAIT)
    assert terminal is None and reward == -120 and done and action == BOOK
    try:
        env.step(BOOK)
    except RuntimeError:
        pass
    else:
        raise AssertionError("Actions after purchase must be rejected")
    env.reset()
    env.state = (6, 240)
    terminal, reward, done, action = env.step(BOOK)
    assert terminal is None and reward == -240 and done and action == BOOK
    env.reset()
    env.state = (0, 180)
    assert env.step(WAIT) == (None, -180, True, BOOK)
    print("PASS P1: boundary, WAIT reward/time, immediate BOOK, deadline BOOK, termination.")

check_topic_1()

## Stop after P1-CHECK

Discuss topic 1 before continuing. The cells below are the intact reference implementation copied from the working Notebook. Do not use Run All while a practice TODO is incomplete. No later-topic exercise has been introduced yet. Official submission features will be implemented separately.

### 2C. Verify the simulator before training
Controlled random choices check every price, every transition day, and all three possible moves, including the boundary stays. Additional episodes check rewards, mandatory deadline booking, and rejection of actions after purchase.

In [ ]:
# 2C — exact transition checks
class ControlledRNG:
    def __init__(self, move):
        self.move = move
        self.seen_probabilities = None

    def choice(self, values, p=None):
        if p is None:
            return 240
        self.seen_probabilities = p
        return self.move

transition_checks = 0
for time_remaining in range(1, 7):
    for price in PRICES:
        for move in (60, 0, -60):
            rng = ControlledRNG(move)
            observed = next_price(time_remaining, int(price), rng)
            expected = min(360, max(120, int(price) + move))
            assert observed == expected
            assert rng.seen_probabilities == ((0.3, 0.3, 0.4) if time_remaining >= 3 else (0.6, 0.2, 0.2))
            transition_checks += 1

In [ ]:
# 2D — rewards, deadline, and termination checks
for purchase_day in range(7):
    env = FlightBookingEnv(np.random.default_rng(100 + purchase_day))
    state = env.reset()
    assert state[0] == 6 and state[1] in PRICES
    for day in range(purchase_day):
        previous = state
        state, reward, done, action = env.step(WAIT)
        assert state[0] == previous[0] - 1 and state[1] in PRICES
        assert reward == 0 and not done and action == WAIT
    price_at_purchase = state[1]
    # Deliberately request WAIT at the deadline to verify forced booking.
    terminal, reward, done, action = env.step(WAIT if purchase_day == 6 else BOOK)
    assert terminal is None and done and action == BOOK
    assert reward == -price_at_purchase
    try:
        env.step(WAIT)
    except RuntimeError:
        pass
    else:
        raise AssertionError("Acting after purchase must fail")
print(f"PASS: {transition_checks} exact transitions; all 7 purchase days; rewards; deadline BOOK; post-purchase rejection.")

## 3. Tabular Q-learning
**Purpose:** learn the cost of BOOK and WAIT from experience. The table has 7 time rows × 5 price rows × 2 actions. At the deadline only BOOK is legal. Greedy ties choose BOOK, making predictions deterministic.

`Q ← old + α × (target − old)`. The target is reward alone for a terminal purchase; otherwise it is reward plus γ times the best legal next-state value. With γ = 1, WAIT includes the undiscounted eventual purchase cost.

In [ ]:
# 3A — indexing, exploration, and greedy prediction
PRICE_INDEX = {int(price): i for i, price in enumerate(PRICES)}

def best_action(q, state):
    time_remaining, price = state
    if time_remaining == 0:
        return BOOK
    return int(np.argmax(q[time_remaining, PRICE_INDEX[price]]))

def epsilon_for(episode_index):
    fraction = min(episode_index / 39_999, 1.0)
    return 1.0 + (0.05 - 1.0) * fraction

def choose_training_action(q, state, rng, epsilon):
    if state[0] == 0:
        return BOOK
    if rng.random() < epsilon:
        return int(rng.integers(2))
    return best_action(q, state)

assert epsilon_for(0) == 1.0
assert np.isclose(epsilon_for(39_999), 0.05)
assert np.isclose(epsilon_for(49_999), 0.05)

In [ ]:
# 3B — one real Q-value update
def update_q(q, state, action, reward, next_state, done, alpha, gamma):
    row = (state[0], PRICE_INDEX[state[1]], action)
    old = float(q[row])
    if done:
        target = float(reward)  # No bootstrap after a purchase.
    else:
        next_action = best_action(q, next_state)
        target = reward + gamma * q[next_state[0], PRICE_INDEX[next_state[1]], next_action]
    q[row] = old + alpha * (target - old)
    return {"state": state, "action": ACTION_NAMES[action], "old": old,
            "reward": reward, "target": float(target), "learning_rate": alpha,
            "updated": float(q[row]), "terminal": done}

# Isolated checks never change the official trained Q-table.
scratch = np.zeros((7, 5, 2))
scratch[0, PRICE_INDEX[240]] = [-240, 999]  # Illegal deadline WAIT must be ignored.
check = update_q(scratch, (1, 240), WAIT, 0, (0, 240), False, 0.1, 1.0)
assert check['target'] == -240 and check['updated'] == -24
check = update_q(scratch, (0, 240), BOOK, -240, None, True, 0.1, 1.0)
assert check['target'] == -240
print("PASS: legal-action bootstrap and terminal reward-only updates.")

### 3C. Train 50,000 episodes
Epsilon decreases linearly from 1.0 on episode 1 to 0.05 on episode 40,000, then stays at 0.05. The learner never reads the probability table. Coverage counts show how often each legal state/action pair was updated; coverage alone does not prove convergence.

In [ ]:
# 3C — training loop
def train_q_learning():
    rng = np.random.default_rng(TRAIN_SEED)
    env = FlightBookingEnv(rng)
    q = np.zeros((7, 5, 2), dtype=float)
    visits = np.zeros_like(q, dtype=int)
    episode_costs = np.empty(TRAIN_EPISODES, dtype=int)
    first_update = None
    wait_update = None
    for episode in range(TRAIN_EPISODES):
        state = env.reset()
        epsilon = epsilon_for(episode)
        while True:
            action = choose_training_action(q, state, rng, epsilon)
            next_state, reward, done, effective_action = env.step(action)
            record = update_q(q, state, effective_action, reward, next_state, done, ALPHA, GAMMA)
            visits[state[0], PRICE_INDEX[state[1]], effective_action] += 1
            if first_update is None:
                first_update = {"episode": episode + 1, **record}
            if wait_update is None and effective_action == WAIT and record['target'] < 0:
                wait_update = {"episode": episode + 1, **record}
            if done:
                episode_costs[episode] = -reward
                break
            state = next_state
    return q, visits, episode_costs, first_update, wait_update

Q, visit_counts, training_costs, first_update, wait_update = train_q_learning()
legal_visits = np.concatenate([visit_counts[0, :, BOOK], visit_counts[1:].ravel()])
assert np.all(legal_visits > 0) and np.all(np.isfinite(Q))
assert np.all(np.isin(training_costs, PRICES))
print(f"Trained {TRAIN_EPISODES:,} complete episodes; all {len(legal_visits)} legal state/action pairs visited.")
print(f"Minimum legal pair visits: {legal_visits.min()}; final epsilon: {epsilon_for(TRAIN_EPISODES - 1):.2f}")

In [ ]:
# 3D — display actual updates and BOOK/WAIT predictions
for label, record in (("First observed update", first_update), ("Observed WAIT update", wait_update)):
    assert record is not None
    print(f"{label}: episode {record['episode']}, state={record['state']}, action={record['action']}")
    print(f"  old={record['old']:.6f}, reward={record['reward']}, target={record['target']:.6f}, alpha={record['learning_rate']}")
    print(f"  updated = {record['old']:.6f} + {record['learning_rate']} * ({record['target']:.6f} - {record['old']:.6f}) = {record['updated']:.6f}")

def predict(time_remaining, current_price):
    if time_remaining not in range(7) or current_price not in PRICE_INDEX:
        raise ValueError("Use time 0–6 and one of the five price levels")
    return ACTION_NAMES[best_action(Q, (time_remaining, current_price))]

for state in ((6, 120), (6, 240), (2, 180), (0, 360)):
    print(f"Prediction for {state}: {predict(*state)}")
assert all(predict(0, int(p)) == "BOOK" for p in PRICES)

## 4. Fair evaluation on 10,000 fresh trajectories
**Purpose:** compare all four policies on the same prices, removing differences caused by unrelated test samples. The simulator pre-generates seven potential prices per trajectory using a separate test seed. These are counterfactual future prices for the evaluator, not inputs to a strategy. Each policy receives **only `(time remaining, current price)`**, and the evaluator stops at purchase.

The Q-learning strategy is greedy: no epsilon, random action, updates, or retraining during evaluation. The official fixed threshold stays at $180.

In [ ]:
# 4A — generate independent shared test trajectories
def generate_test_trajectories(count, seed):
    env = FlightBookingEnv(np.random.default_rng(seed))
    paths = np.empty((count, 7), dtype=int)
    for index in range(count):
        state = env.reset()
        paths[index, 0] = state[1]
        for day in range(1, 7):
            state, reward, done, action = env.step(WAIT)
            assert reward == 0 and not done
            paths[index, day] = state[1]
    paths.setflags(write=False)
    return paths

test_paths = generate_test_trajectories(TEST_TRAJECTORIES, TEST_SEED)
assert test_paths.shape == (10_000, 7)
assert np.all(np.isin(test_paths, PRICES))
print(f"Generated {len(test_paths):,} fresh seven-price trajectories with test seed {TEST_SEED}.")

In [ ]:
# 4B — state-only policies
def immediate_policy(state):
    return BOOK

def threshold_policy(threshold):
    def policy(state):
        time_remaining, price = state
        return BOOK if time_remaining == 0 or price <= threshold else WAIT
    return policy

def learned_policy(state):
    return best_action(Q, state)  # Deterministic; reads no future prices.

In [ ]:
# 4C — stop-on-purchase evaluator
def play_path(path, policy):
    trace = []
    for day in range(7):
        state = (6 - day, int(path[day]))
        requested_action = policy(state)
        if requested_action not in (BOOK, WAIT):
            raise ValueError("Policy returned an invalid action")
        action = BOOK if state[0] == 0 else requested_action
        reward = -state[1] if action == BOOK else 0
        trace.append({"day": day + 1, "state": state,
                      "action": ACTION_NAMES[action], "reward": reward})
        if action == BOOK:
            return state[1], trace
    raise AssertionError("Deadline booking must terminate the trajectory")

OFFICIAL_POLICIES = {
    "Immediate booking": immediate_policy,
    "Minimum-price rule ($120)": threshold_policy(120),
    "Fixed-threshold rule ($180)": threshold_policy(OFFICIAL_THRESHOLD),
    "Q-learning (greedy)": learned_policy,
}
assert OFFICIAL_THRESHOLD == 180

In [ ]:
# 4D — evaluator checks
# Policy-call counts verify termination and state-only observations.
seen_states = []
def spy_policy(state):
    assert isinstance(state, tuple) and len(state) == 2
    seen_states.append(state)
    return threshold_policy(180)(state)
synthetic = [300, 240, 180, 120, 360, 360, 360]
cost, trace = play_path(synthetic, spy_policy)
assert cost == 180 and seen_states == [(6, 300), (5, 240), (4, 180)]
assert play_path(synthetic, immediate_policy)[0] == 300
assert play_path(synthetic, threshold_policy(120))[0] == 120
cost, trace = play_path([360] * 7, lambda state: WAIT)
assert cost == 360 and len(trace) == 7 and trace[-1]['action'] == 'BOOK'
print("PASS: current-state-only calls, all baseline rules, stop after purchase, forced deadline in evaluator.")

In [ ]:
# 4E — official comparison and paired cost differences
q_before_evaluation = Q.copy()
official_costs = {
    name: np.array([play_path(path, policy)[0] for path in test_paths], dtype=int)
    for name, policy in OFFICIAL_POLICIES.items()
}
assert np.array_equal(Q, q_before_evaluation)
assert all(len(costs) == 10_000 and np.all(np.isin(costs, PRICES)) for costs in official_costs.values())
q_costs = official_costs['Q-learning (greedy)']
comparison = []
for name, costs in official_costs.items():
    comparison.append({"strategy": name, "trajectories": len(costs),
                       "mean_purchase_cost": float(costs.mean()),
                       "mean_paired_saving_vs_q": float((costs - q_costs).mean())})
table = "| Strategy | Test trajectories | Mean purchase cost | Mean paired saving from Q |\n|---|---:|---:|---:|\n"
for row in comparison:
    table += f"| {row['strategy']} | {row['trajectories']:,} | ${row['mean_purchase_cost']:.2f} | ${row['mean_paired_saving_vs_q']:.2f} |\n"
display(Markdown(table))
print("Positive saving = Q-learning costs less; negative saving = baseline costs less.")
print("PASS: evaluation made no changes to the trained Q-table; exploration disabled.")

## 5. One complete Q-learning decision trace
**Purpose:** connect current state → greedy action → observed reward → final purchase cost. For a readable example, choose the first test trajectory with at least three decisions. This selection is only for illustration, never for computing the comparison. Stop displaying at purchase; unseen future prices are not shown.

In [ ]:
# 5A — actual states, actions, and final cost
trace_index = next(i for i, path in enumerate(test_paths)
                   if len(play_path(path, learned_policy)[1]) >= 3)
trace_cost, decision_trace = play_path(test_paths[trace_index], learned_policy)
trace_table = "| Day | State (time remaining, price) | Q(BOOK) | Q(WAIT) | Action | Reward |\n|---:|---|---:|---:|---|---:|\n"
for step in decision_trace:
    time_remaining, price = step['state']
    book_value, wait_value = Q[time_remaining, PRICE_INDEX[price]]
    wait_text = f"{wait_value:.3f}" if time_remaining > 0 else "illegal"
    trace_table += f"| {step['day']} | ({time_remaining}, ${price}) | {book_value:.3f} | {wait_text} | {step['action']} | {step['reward']} |\n"
display(Markdown(trace_table))
print(f"Test trajectory index (zero-based): {trace_index}")
print(f"Final purchase cost: ${trace_cost}; episode ended after {len(decision_trace)} decisions.")
assert decision_trace[-1]['action'] == 'BOOK'
assert all(step['action'] == 'WAIT' for step in decision_trace[:-1])
assert sum(step['reward'] for step in decision_trace) == -trace_cost

## 6. Separate practice exercise: change a threshold
**Purpose:** practice how a simple rule affects cost. Change only `PRACTICE_THRESHOLD` in 6A to 120, 180, 240, 300, or 360, then rerun 6A. It reuses the test trajectories for exploration of the model; it is not a new held-out evaluation. Never choose or tune the official threshold from these results.

The official four-strategy costs and threshold remain intact. This cell was executed by the implementation agent; it does **not** establish that you personally practiced it.

In [ ]:
# 6A — editable practice threshold (official comparison unchanged)
PRACTICE_THRESHOLD = 240  # CHANGE THIS VALUE, then rerun this cell.
if PRACTICE_THRESHOLD not in PRICE_INDEX:
    raise ValueError("Choose 120, 180, 240, 300, or 360")
official_snapshot = {name: costs.copy() for name, costs in official_costs.items()}
practice_costs = np.array([play_path(path, threshold_policy(PRACTICE_THRESHOLD))[0]
                          for path in test_paths])
print(f"PRACTICE ONLY: threshold ${PRACTICE_THRESHOLD}; mean cost ${practice_costs.mean():.2f}")
print(f"Official threshold remains ${OFFICIAL_THRESHOLD}; official mean is ${official_costs['Fixed-threshold rule ($180)'].mean():.2f}")
assert OFFICIAL_THRESHOLD == 180
assert all(np.array_equal(official_costs[name], costs) for name, costs in official_snapshot.items())

## 7. Save actual results and verification evidence
**Purpose:** keep the mean comparison, all paired costs, test trajectories, learned Q-values, observed updates, and trace available for review. The executed Notebook itself stores the printed and displayed outputs. Rerunning writes fresh artifacts from that execution.

In [ ]:
# 7A — final verification
# Future changes after purchase must never change decisions or cost.
for name, policy in OFFICIAL_POLICIES.items():
    for path in test_paths[:100]:
        cost, trace = play_path(path, policy)
        altered = path.copy()
        altered[len(trace):] = 120
        assert play_path(altered, policy) == (cost, trace)
assert np.array_equal(Q, q_before_evaluation)

In [ ]:
# 7B — save paired costs and arrays
RESULTS = Path('practice_results')
RESULTS.mkdir(exist_ok=True)
with (RESULTS / 'mean_cost_comparison.csv').open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)
with (RESULTS / 'paired_purchase_costs.csv').open('w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['trajectory_index', *OFFICIAL_POLICIES])
    for i in range(TEST_TRAJECTORIES):
        writer.writerow([i, *(official_costs[name][i] for name in OFFICIAL_POLICIES)])
np.save(RESULTS / 'test_trajectories.npy', test_paths)
np.save(RESULTS / 'q_values.npy', Q)
np.save(RESULTS / 'training_visit_counts.npy', visit_counts)

In [ ]:
# 7C — save actual run summary and trace
summary = {
    'training_seed': TRAIN_SEED, 'test_seed': TEST_SEED,
    'training_episodes': TRAIN_EPISODES, 'test_trajectories': TEST_TRAJECTORIES,
    'alpha': ALPHA, 'gamma': GAMMA, 'epsilon_start': 1.0, 'epsilon_end': 0.05,
    'epsilon_decay_episodes': 40_000, 'evaluation_exploration': 0,
    'official_threshold': OFFICIAL_THRESHOLD, 'comparison': comparison,
    'first_update': first_update, 'observed_wait_update': wait_update,
    'trace_index': trace_index, 'decision_trace': decision_trace,
    'final_purchase_cost': trace_cost,
    'verification': {'exact_transition_cases': transition_checks,
                     'purchase_days_checked': 7, 'legal_pairs_visited': int((legal_visits > 0).sum()),
                     'minimum_legal_pair_visits': int(legal_visits.min()),
                     'terminal_updates_reward_only': True, 'deadline_booking': True,
                     'post_purchase_rejected': True, 'q_unchanged_during_evaluation': True,
                     'future_suffix_invariance_cases': 400},
    'execution': {'python': sys.executable, 'python_version': sys.version.split()[0],
                  'numpy_version': np.__version__},
    'personally_practiced_by_user': 'Not established; user must run and explain the demo cells.'
}
(RESULTS / 'run_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
(RESULTS / 'official_comparison.md').write_text(table + '\nPositive paired saving = baseline cost minus Q-learning cost.\n')
(RESULTS / 'q_learning_trace.md').write_text(trace_table + f'\nTrajectory {trace_index}; final purchase cost: ${trace_cost}.\n')
print('PASS: 400 future-suffix invariance cases; all core checks passed.')
print('Saved actual comparison, 10,000 paired costs, price trajectories, Q-table, visits, updates, and complete trace in results/.')

## 8. Interpretation and limits
**Purpose:** explain actual observations without claiming convergence or universal savings. The following interpretation is calculated from the outputs of this execution.

In [ ]:
# 8A — evidence-based interpretation
best_row = min(comparison, key=lambda row: row['mean_purchase_cost'])
fixed_saving = (official_costs['Fixed-threshold rule ($180)'] - q_costs).mean()
print(f"Lowest observed mean: {best_row['strategy']}, ${best_row['mean_purchase_cost']:.2f}.")
print(f"Mean paired saving from Q-learning versus the official $180 rule: ${fixed_saving:.3f}.")
print(f"The displayed Q-learning trajectory still costs ${trace_cost}; average savings do not imply savings on every trip.")
print('These results cover one training seed and one test sample from hypothetical probabilities.')
print('Fixed learning rate and finite training do not guarantee convergence. No real-flight recommendation is supported.')
print('Deferred: 95% confidence intervals, exact finite-horizon MDP reference, and policy heatmap.')

## 9. Essential cells to personally run and explain before recording
The implementation agent wrote and executed the core, including the default practice example. **Your personal practice has not been observed or recorded.** Before making the approximately 10-minute video required by the assignment, select the project kernel and restart/run the entire Notebook yourself.

Use these labeled cells as your explanation checklist:

1. **1A:** show the interpreter, independent seeds, α = 0.1, γ = 1, 50,000 training episodes, and 10,000 test paths.
2. **2A–2D:** explain the two probability regimes, boundary stays, negative purchase reward, zero WAIT reward, seven opportunities, mandatory deadline booking, and termination checks.
3. **3A–3D:** explain state indexing, epsilon-greedy training versus greedy predictions, terminal versus nonterminal targets, and the actual old/target/learning-rate/updated Q-values.
4. **4A–4E:** explain why every strategy gets the same trajectories, how state-only calls prevent future access, why testing has no exploration, and how mean paired saving is calculated.
5. **5A:** narrate the complete trace until purchase, including the final cost and why a single trip may be expensive.
6. **6A:** personally change the practice threshold (for example to 120), rerun, and explain the changed practice result while the official $180 comparison stays fixed.
7. **7A–7C and 8A:** show the saved evidence, explain observed results and limits, and name the deferred analyses.

The unchanged proposal and course PDF remain in the project folder. This core Notebook and README do not replace the separate final PDF/Word proposal/writeup or your camera-on recording required by the assignment.